# 01 Data cleaning

**Topic:** Resident Satisfaction with Local Services in Tilak Nagar, Mumbai.

Run each cell from top to bottom. This notebook reads the raw survey, cleans it, and saves two CSV files. Every important step is visible here; there is no helper file to learn.

A **variable** is a name for a value. A **DataFrame** is a table. A **dictionary** pairs an old name with a new name. A **for loop** repeats the same step for each column.

## 1. Import tools and find the project folder
This setup works when the notebook starts in the project folder or the notebooks folder.

In [1]:
# Pandas helps us work with tables.
import pandas as pd

# os helps us check where the data folder is.
import os

# Start by looking in the project folder.
project_folder = "."

# A notebook may start inside the notebooks folder instead.
if not os.path.exists("data/raw/survey_responses_raw.csv"):

    # Two dots mean the folder one level above this one.
    project_folder = ".."

## 2. Read the raw survey
Read everything as text at first. Keep blank answers as empty text. The original CSV will stay unchanged.

In [2]:
# Read the original survey file.
data = pd.read_csv(project_folder + "/data/raw/survey_responses_raw.csv", dtype=str, keep_default_na=False)

# Count the supplied rows.
print("Raw responses:", len(data))

# Show the first five rows.
data.head()

Raw responses: 32


,Timestamp,1. Do you agree to participate in this survey?,2. Which locality do you live in?,3. What is your age group?,4. How long have you lived in this area?,5. How satisfied are you with the regularity of water supply?,6. How satisfied are you with garbage collection?,7. How satisfied are you with the cleanliness of public areas?,8. How satisfied are you with the condition of roads and footpaths?,9. How satisfied are you with street lighting in your area?,10. How satisfied are you with drainage and sewage services?,11. How satisfied are you with the availability and cleanliness of public toilets?,12. How satisfied are you with online municipal services or complaint systems?,13. Have you ever reported a local civic problem?,"14. Overall, how satisfied are you with local government services in your area?","15. Which local service needs the most urgent improvement, and what is your suggestion?"
0,2026/10/02 1:38:44 pm GMT+5:30,Yes,Tilak Nagar,18–30,More than 7 years,3 - Neutral,2 - Dissatisfied,3 - Neutral,3 - Neutral,2 - Dissatisfied,2 - Dissatisfied,1 - Very dissatisfied,3 - Neutral,No,3,In Tilak nagar we need parking place\n
1,2026/10/02 1:41:59 pm GMT+5:30,Yes,Other,18–30,More than 7 years,3 - Neutral,3 - Neutral,2 - Dissatisfied,2 - Dissatisfied,2 - Dissatisfied,2 - Dissatisfied,3 - Neutral,2 - Dissatisfied,No,3,roads
2,2026/10/02 1:45:42 pm GMT+5:30,Yes,Tilak Nagar,18–30,1–3 years,2 - Dissatisfied,2 - Dissatisfied,3 - Neutral,3 - Neutral,2 - Dissatisfied,4 - Satisfied,3 - Neutral,4 - Satisfied,I do not know how to report one,3,
3,2026/10/02 3:19:15 pm GMT+5:30,Yes,Tilak Nagar,18–30,4–7 years,3 - Neutral,3 - Neutral,3 - Neutral,4 - Satisfied,4 - Satisfied,5 - Very satisfied,4 - Satisfied,5 - Very satisfied,Yes,5,
4,2026/10/02 3:44:04 pm GMT+5:30,Yes,Chembur,18–30,More than 7 years,4 - Satisfied,3 - Neutral,1 - Very dissatisfied,3 - Neutral,4 - Satisfied,3 - Neutral,1 - Very dissatisfied,I have not used them,No,1,We need thee cleanness in aur public toilet in...


## 3. Remove exact duplicate rows
An exact duplicate has the same values in every column, including its timestamp. Matching answers alone do not prove a duplicate person.

In [3]:
# Count completely identical rows.
print("Exact duplicate rows:", data.duplicated().sum())

# Keep one copy of each completely identical row.
data = data.drop_duplicates()

# Remove spaces from the beginning and end of column headings.
data.columns = data.columns.str.strip()

Exact duplicate rows: 0


## 4. Shorten the column names
Each line in this dictionary means **old question: new column name**. Matching the question text keeps the code correct even if the CSV columns move.

In [4]:
# Give each long survey question a short column name.
column_names = {
    'Timestamp': 'timestamp',
    '1. Do you agree to participate in this survey?': 'consent',
    '2. Which locality do you live in?': 'locality',
    '3. What is your age group?': 'age_group',
    '4. How long have you lived in this area?': 'years_in_area',
    '5. How satisfied are you with the regularity of water supply?': 'water_supply',
    '6. How satisfied are you with garbage collection?': 'garbage_collection',
    '7. How satisfied are you with the cleanliness of public areas?': 'public_cleanliness',
    '8. How satisfied are you with the condition of roads and footpaths?': 'roads_footpaths',
    '9. How satisfied are you with street lighting in your area?': 'street_lighting',
    '10. How satisfied are you with drainage and sewage services?': 'drainage',
    '11. How satisfied are you with the availability and cleanliness of public toilets?': 'public_toilets',
    '12. How satisfied are you with online municipal services or complaint systems?': 'online_services',
    '13. Have you ever reported a local civic problem?': 'reported_problem',
    '14. Overall, how satisfied are you with local government services in your area?': 'overall_satisfaction',
    '15. Which local service needs the most urgent improvement, and what is your suggestion?': 'suggestion',
}

# Rename the columns using the dictionary above.
data = data.rename(columns=column_names)

# Show the short column names.
print(data.columns.tolist())

['timestamp', 'consent', 'locality', 'age_group', 'years_in_area', 'water_supply', 'garbage_collection', 'public_cleanliness', 'roads_footpaths', 'street_lighting', 'drainage', 'public_toilets', 'online_services', 'reported_problem', 'overall_satisfaction', 'suggestion']


## 5. Clean text and check consent
A for loop lets us remove spaces from every text column without writing the same command sixteen times.

In [5]:
# Repeat the cleaning step for each column.
for column in data.columns:

    # Remove spaces before and after each answer.
    data[column] = data[column].str.strip()

# Keep only people who answered Yes to participation.
data = data[data["consent"].str.lower() == "yes"].copy()

# Use the same capital letters for each locality.
data["locality"] = data["locality"].str.title()

# Use a normal hyphen in age ranges.
data["age_group"] = data["age_group"].str.replace("–", "-", regex=False)

# Keep the original online-service answer before changing it to a number.
data["online_services_response"] = data["online_services"]

## 6. Explain the rating scale
The service answers include text such as `4 - Satisfied`. We first keep just the numeric text, such as `"4"`. The next step converts it into a number. `None` means no numeric answer; it is **not zero**.

In [6]:
# Match each full rating answer to its numeric text.
rating_numbers = {
    "1 - Very dissatisfied": "1",
    "2 - Dissatisfied": "2",
    "3 - Neutral": "3",
    "4 - Satisfied": "4",
    "5 - Very satisfied": "5",
    "Not applicable": None,
    "I have not used them": None,
    "": None,
}

# List the columns that contain satisfaction ratings.
rating_columns = [
    "water_supply",
    "garbage_collection",
    "public_cleanliness",
    "roads_footpaths",
    "street_lighting",
    "drainage",
    "public_toilets",
    "online_services",
    "overall_satisfaction",
]

## 7. Convert ratings into numbers
`pd.to_numeric` converts numeric text such as `"3"` into a number. An unexpected word causes an error so we can check it. The extra check rejects numbers outside 1 to 5.

In [7]:
# Repeat these steps for each rating column.
for column in rating_columns:

    # Replace the full rating answers using our dictionary.
    data[column] = data[column].replace(rating_numbers)

    # Convert the remaining numeric text into numbers.
    data[column] = pd.to_numeric(data[column])

    # Leave out missing answers when checking the scale.
    valid_answers = data[column].dropna()

    # Check that every recorded rating is 1, 2, 3, 4, or 5.
    if not valid_answers.isin([1, 2, 3, 4, 5]).all():

        # Stop instead of calculating with an incorrect rating.
        raise ValueError("Check the ratings in " + column)

## 8. Select the study area
Keep all consenting responses in one file. Save the Tilak Nagar responses separately for the main analysis.

In [8]:
# Keep the rows for the project locality.
tilak_data = data[data["locality"] == "Tilak Nagar"].copy()

# Show the sample sizes.
print("All consenting responses:", len(data))
print("Tilak Nagar responses:", len(tilak_data))
print("Other locality responses:", len(data) - len(tilak_data))

# Count valid ratings for every service.
print(tilak_data[rating_columns].count())

All consenting responses: 32
Tilak Nagar responses: 26
Other locality responses: 6
water_supply            26
garbage_collection      26
public_cleanliness      26
roads_footpaths         26
street_lighting         26
drainage                25
public_toilets          26
online_services         12
overall_satisfaction    26
dtype: int64


## 9. Save the cleaned files
`index=False` means do not add the table row numbers as another CSV column.

In [9]:
# Create the output folder if it does not exist.
os.makedirs(project_folder + "/data/processed", exist_ok=True)

# Save all consenting responses.
data.to_csv(project_folder + "/data/processed/responses_clean.csv", index=False)

# Save the Tilak Nagar responses.
tilak_data.to_csv(project_folder + "/data/processed/tilak_nagar_clean.csv", index=False)

# Confirm that cleaning is complete.
print("Saved the two cleaned CSV files. Open notebook 02 next.")

Saved the two cleaned CSV files. Open notebook 02 next.


## What to tell your teacher

“I read the survey, removed exact duplicate rows, shortened the column names, kept consenting responses, changed rating text into numbers, selected Tilak Nagar, and saved the results.”

For the supplied file: **32** responses, **26** from Tilak Nagar, and **0** exact duplicate rows. Drainage has **25** valid ratings. Online services have **12** valid ratings and **14** explicit non-use answers. Missing ratings are excluded from averages.

Two Tilak Nagar respondents reported an age below 18. Guardian-permission details were not supplied; discuss use of these records with your supervisor. The sample is small and recruitment details are unknown, so findings describe these respondents.

If the data changes, run this notebook and then notebooks 02 and 03 again. Review the written findings and report separately.